# 05 — Remaining preliminary runs (1 seed, untuned defaults)
Feature ablation (M1 on MFCC), leakage diagnostic for M2 and M3, and M4 on all 6 folds.
All runs use tag `prelim`; logs in `results/logs/`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

project root: <repo>


In [2]:
import subprocess, time
LOGS = ROOT / "results" / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def run(name, *args):
    """Run `python -m <args>` from 02_code, stream output here and to results/logs/<name>.log."""
    cmd = [sys.executable, "-u", "-m", *args]
    t0 = time.time()
    with open(LOGS / f"{name}.log", "w", encoding="utf-8") as log:
        log.write(" ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace")
        finished = False
        for line in p.stdout:
            print(line, end="")
            log.write(line); log.flush()
            finished = finished or line.startswith("[done] exit code 0")
        p.wait()
        log.write(f"\n[exit {p.returncode}] {time.time() - t0:.0f} s\n")
    print(f"\n[{name}] exit {p.returncode} in {time.time() - t0:.0f} s")
    # Windows sometimes crashes during interpreter shutdown (0xC0000409) AFTER all results are written.
    # A step counts as successful only if (a) it exited 0 or reported "[done] exit code 0", AND
    # (b) every run it started has complete outputs on disk (runs.csv row, curves, 240 test predictions).
    from src import verify_runs
    v = verify_runs.verify_log(LOGS / f"{name}.log")
    print(v["summary"])
    if (p.returncode != 0 and not finished) or not v["ok"]:
        raise RuntimeError(f"{name} failed -> {LOGS / (name + '.log')}")

## Feature ablation: M1 on 40 MFCC + delta + delta-delta

In [3]:
run("prelim_m1_mfcc", "src.run_experiments", "--model", "m1", "--features", "mfcc", "--folds", "0-5", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m1_mfcc_actor_f0_s0_prelim | cuda | params 653,576 (trainable 653,576) | cfg 65c4b81c8c
[m1_mfcc_actor_f0_s0_prelim] ep   1 lr 1.00e-03 | train loss 2.023 acc 0.190 | val loss 1.978 UAR 0.203 *
[m1_mfcc_actor_f0_s0_prelim] ep   2 lr 1.00e-03 | train loss 1.877 acc 0.300 | val loss 1.826 UAR 0.281 *
[m1_mfcc_actor_f0_s0_prelim] ep   3 lr 1.00e-03 | train loss 1.804 acc 0.343 | val loss 1.770 UAR 0.340 *
[m1_mfcc_actor_f0_s0_prelim] ep   4 lr 1.00e-03 | train loss 1.661 acc 0.446 | val loss 1.649 UAR 0.391 *
[m1_mfcc_actor_f0_s0_prelim] ep   5 lr 1.00e-03 | train loss 1.520 acc 0.504 | val loss 1.600 UAR 0.418 *
[m1_mfcc_actor_f0_s0_prelim] ep   6 lr 1.00e-03 | train loss 1.408 acc 0.562 | val loss 1.564 UAR 0.449 *
[m1_mfcc_actor_f0_s0_prelim] ep   7 lr 1.00e-03 | train loss 1.377 acc 0.562 | val loss 1.586 UAR 0.445
[m1_mfcc_actor_f0_s0_prelim] ep   8 lr 1.00e-03 | train loss 1.295 acc 0.631 | val loss 1.554 UAR 0.461 *
[m1_mfcc_actor_f0_s0_prelim] ep   9 lr 1.00e-03 | train loss 1.

## Leakage diagnostic: random utterance-level split for M2 and M3

In [4]:
run("prelim_m2_random", "src.run_experiments", "--model", "m2", "--protocol", "random", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m2_logmel_random_fR_s0_prelim | cuda | params 1,276,904 (trainable 1,276,904) | cfg e008d30dd9
[m2_logmel_random_fR_s0_prelim] ep   1 lr 1.00e-03 | train loss 2.033 acc 0.184 | val loss 1.864 UAR 0.285 *
[m2_logmel_random_fR_s0_prelim] ep   2 lr 1.00e-03 | train loss 1.894 acc 0.274 | val loss 1.749 UAR 0.402 *
[m2_logmel_random_fR_s0_prelim] ep   3 lr 1.00e-03 | train loss 1.820 acc 0.352 | val loss 1.740 UAR 0.309
[m2_logmel_random_fR_s0_prelim] ep   4 lr 1.00e-03 | train loss 1.773 acc 0.357 | val loss 1.664 UAR 0.406 *
[m2_logmel_random_fR_s0_prelim] ep   5 lr 1.00e-03 | train loss 1.757 acc 0.358 | val loss 1.630 UAR 0.422 *
[m2_logmel_random_fR_s0_prelim] ep   6 lr 1.00e-03 | train loss 1.703 acc 0.399 | val loss 1.549 UAR 0.480 *
[m2_logmel_random_fR_s0_prelim] ep   7 lr 1.00e-03 | train loss 1.654 acc 0.435 | val loss 1.536 UAR 0.484 *
[m2_logmel_random_fR_s0_prelim] ep   8 lr 1.00e-03 | train loss 1.656 acc 0.417 | val loss 1.498 UAR 0.531 *
[m2_logmel_random_fR_s0_prelim] 

In [5]:
run("prelim_m3_random", "src.run_experiments", "--model", "m3", "--protocol", "random", "--seeds", "0", "--tag", "prelim", "--skip-existing")

== m3_logmel_random_fR_s0_prelim | cuda | params 2,423,048 (trainable 2,423,048) | cfg 97b7549749
[m3_logmel_random_fR_s0_prelim] ep   1 lr 1.00e-04 | train loss 2.138 acc 0.142 | val loss 2.038 UAR 0.141 *
[m3_logmel_random_fR_s0_prelim] ep   2 lr 2.00e-04 | train loss 2.043 acc 0.189 | val loss 1.926 UAR 0.332 *
[m3_logmel_random_fR_s0_prelim] ep   3 lr 3.00e-04 | train loss 1.944 acc 0.266 | val loss 1.871 UAR 0.332
[m3_logmel_random_fR_s0_prelim] ep   4 lr 4.00e-04 | train loss 1.910 acc 0.285 | val loss 1.727 UAR 0.375 *
[m3_logmel_random_fR_s0_prelim] ep   5 lr 5.00e-04 | train loss 1.886 acc 0.289 | val loss 1.803 UAR 0.316
[m3_logmel_random_fR_s0_prelim] ep   6 lr 5.00e-04 | train loss 1.830 acc 0.315 | val loss 1.670 UAR 0.371
[m3_logmel_random_fR_s0_prelim] ep   7 lr 5.00e-04 | train loss 1.813 acc 0.341 | val loss 1.646 UAR 0.449 *
[m3_logmel_random_fR_s0_prelim] ep   8 lr 5.00e-04 | train loss 1.799 acc 0.327 | val loss 1.599 UAR 0.445
[m3_logmel_random_fR_s0_prelim] ep   9

## M4 (wav2vec 2.0 Base) on all 6 folds

In [6]:
run("prelim_m4_all", "src.run_experiments", "--model", "m4", "--folds", "0-5", "--seeds", "0", "--tag", "prelim", "--skip-existing")

skip m4_wave_actor_f0_s0_prelim (already in runs.csv)
W1121 18:21:54.625000 5960 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 42231.25it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
== m4_wave_actor_f1_s0_prelim | cuda | params 94,377,864 (trainable 90,177,416) | c